In [ ]:
import numpy as np
import pandas as pd
from scipy.io import arff
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
import time

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

os.makedirs('../data', exist_ok=True)
os.makedirs('../output', exist_ok=True)
os.makedirs('../visualizations', exist_ok=True)

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

In [ ]:
ARFF_PATH = '../data/OVA_Breast.arff'

data, meta = arff.loadarff(ARFF_PATH)
df_raw = pd.DataFrame(data)

# Kategoricke/string kolone se u scipy.io.arff ucitavaju kao bytes objekti - dekodiramo ih
df_raw['Tissue'] = df_raw['Tissue'].str.decode('utf-8')

print(f"Oblik podataka: {df_raw.shape}")
print(f"Kolone (prvih 5): {df_raw.columns[:5].tolist()}")
print(f"Kolone (poslednje 3): {df_raw.columns[-3:].tolist()}")

In [ ]:
print("=== Osnovne informacije ===")
print(f"Broj instanci: {df_raw.shape[0]}")
print(f"Broj atributa (ukupno, sa ID_REF i Tissue): {df_raw.shape[1]}")
print(f"Broj gen-ekspresija atributa: {df_raw.shape[1] - 2}")
print()
print("=== Raspodela ciljnog atributa Tissue ===")
print(df_raw['Tissue'].value_counts())
print()
print(df_raw['Tissue'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
counts = df_raw['Tissue'].value_counts()
colors = ['#D85A30', '#378ADD']
ax.bar(counts.index, counts.values, color=colors)
ax.set_ylabel('Broj uzoraka')
ax.set_title('Raspodela klasa - Tissue (samo informativno,\nne koristi se u klasterovanju)')
for i, v in enumerate(counts.values):
    ax.text(i, v + 15, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../visualizations/class_distribution.png', dpi=150)
plt.show()

print(f"\nNapomena: klase su neuravnotezene ({counts.iloc[0]} vs {counts.iloc[1]}, "
      f"odnos ~{counts.iloc[0]/counts.iloc[1]:.1f}:1). Ovo je bitno pri tumacenju "
      f"eksterne evaluacije klastera - klasteri se ne moraju poklapati 1:1 sa ovom podelom.")

In [ ]:
missing = df_raw.isnull().sum().sum()
print(f"Ukupan broj nedostajucih vrednosti: {missing}")

dup_ids = df_raw['ID_REF'].duplicated().sum()
print(f"Broj dupliranih ID_REF vrednosti: {dup_ids}")

dup_rows = df_raw.drop(columns=['ID_REF']).duplicated().sum()
print(f"Broj potpuno dupliranih redova (bez ID_REF): {dup_rows}")

sample_cols = df_raw.drop(columns=['ID_REF', 'Tissue']).sample(5, axis=1, random_state=RANDOM_STATE).columns
print("\nRaspon vrednosti za 5 nasumicno izabranih gena (ilustracija heterogenosti):")
print(df_raw[sample_cols].describe().loc[['min', 'mean', 'max']].T)


In [ ]:
y_tissue = df_raw['Tissue'].copy()
X_raw = df_raw.drop(columns=['ID_REF', 'Tissue'])

print(f"Oblik matrice atributa (X): {X_raw.shape}")
print(f"Oblik labele (y, samo za evaluaciju): {y_tissue.shape}")


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
X_scaled = pd.DataFrame(X_scaled, columns=X_raw.columns, index=X_raw.index)

print("Standardizacija zavrsena.")
print(f"Prosek po koloni posle skaliranja (treba biti ~0): {X_scaled.values.mean():.6f}")
print(f"Std po koloni posle skaliranja (treba biti ~1): {X_scaled.values.std():.6f}")

In [ ]:
df_preprocessed = X_scaled.copy()
df_preprocessed['Tissue'] = y_tissue.values

df_preprocessed.to_pickle('../data/data_preprocessed_full.pkl')
size_mb = os.path.getsize('../data/data_preprocessed_full.pkl') / (1024**2)
print(f"Sacuvano: data/data_preprocessed_full.pkl, oblik: {df_preprocessed.shape}, velicina: {size_mb:.1f} MB")

In [ ]:
from sklearn.decomposition import PCA

pca_full = PCA(random_state=RANDOM_STATE).fit(X_scaled)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)

for target in [0.5, 0.7, 0.9, 0.95]:
    n_comp = np.argmax(cumvar >= target) + 1
    print(f"{target*100:.0f}% objasnjene varijanse -> potrebno {n_comp} komponenti")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, len(cumvar) + 1), cumvar, color='#0F6E56')
ax.axhline(0.9, color='#993C1D', linestyle='--', linewidth=1, label='90% varijanse')
ax.set_xlabel('Broj glavnih komponenti')
ax.set_ylabel('Kumulativna objasnjena varijansa')
ax.set_title('PCA - kumulativna objasnjena varijansa')
ax.legend()
plt.tight_layout()
plt.savefig('../visualizations/pca_explained_variance.png', dpi=150)
plt.show()

In [ ]:
N_COMPONENTS = 50
pca_50 = PCA(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
X_pca50 = pca_50.fit_transform(X_scaled)

print(f"PCA (50 komponenti) oblik: {X_pca50.shape}")
print(f"Objasnjena varijansa: {pca_50.explained_variance_ratio_.sum()*100:.2f}%")

df_pca50 = pd.DataFrame(X_pca50, columns=[f'PC{i+1}' for i in range(N_COMPONENTS)])
df_pca50['Tissue'] = y_tissue.values
df_pca50.to_csv('../data/data_preprocessed_pca50.csv', index=False)
print("Sacuvano: data/data_preprocessed_pca50.csv")


In [ ]:
TOP_K = 200
variances = X_scaled.var(axis=0)
top_genes = variances.sort_values(ascending=False).head(TOP_K).index

X_topvar = X_scaled[top_genes]
print(f"Skup top-{TOP_K} najvarijabilnijih gena, oblik: {X_topvar.shape}")

df_topvar = X_topvar.copy()
df_topvar['Tissue'] = y_tissue.values
df_topvar.to_csv('../data/data_preprocessed_topvar200.csv', index=False)
print(f"Sacuvano: data/data_preprocessed_topvar200.csv")


In [ ]:
from mpl_toolkits.mplot3d import Axes3D

pca_3 = PCA(n_components=3, random_state=RANDOM_STATE)
X_pca3 = pca_3.fit_transform(X_scaled)

colors_map = {'Breast': '#D85A30', 'Other': '#378ADD'}

fig = plt.figure(figsize=(13, 5))

ax1 = fig.add_subplot(1, 2, 1)
for label, color in colors_map.items():
    mask = (y_tissue == label).values
    ax1.scatter(X_pca3[mask, 0], X_pca3[mask, 1], c=color, label=label, alpha=0.6, s=15)
ax1.set_xlabel('PC1')
ax1.set_ylabel('PC2')
ax1.set_title('2D PCA projekcija (obojeno po Tissue)')
ax1.legend()

ax2 = fig.add_subplot(1, 2, 2, projection='3d')
for label, color in colors_map.items():
    mask = (y_tissue == label).values
    ax2.scatter(X_pca3[mask, 0], X_pca3[mask, 1], X_pca3[mask, 2], c=color, label=label, alpha=0.6, s=15)
ax2.set_xlabel('PC1')
ax2.set_ylabel('PC2')
ax2.set_zlabel('PC3')
ax2.set_title('3D PCA projekcija (obojeno po Tissue)')
ax2.legend()

plt.tight_layout()
plt.savefig('../visualizations/pca_2d_3d.png', dpi=150)
plt.show()

In [ ]:
with open('../output/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open('../output/pca_50.pkl', 'wb') as f:
    pickle.dump(pca_50, f)
with open('../output/top_var_genes.pkl', 'wb') as f:
    pickle.dump(list(top_genes), f)

print("Sacuvani pomocni objekti: scaler.pkl, pca_50.pkl, top_var_genes.pkl")
print("\nDAN 1 zavrsen. Spremno za Dan 2 - klasterovanje.")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import time
import os

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

df_pca50 = pd.read_csv('../data/data_preprocessed_pca50.csv')
df_topvar = pd.read_csv('../data/data_preprocessed_topvar200.csv')

y_true = df_pca50['Tissue'].values
X_pca50 = df_pca50.drop(columns=['Tissue']).values
X_topvar = df_topvar.drop(columns=['Tissue']).values

print("PCA-50 shape:", X_pca50.shape)
print("TopVar-200 shape:", X_topvar.shape)

In [ ]:
from scipy.io import arff

data, meta = arff.loadarff('../data/OVA_Breast.arff')
df_raw = pd.DataFrame(data)
df_raw['Tissue'] = df_raw['Tissue'].str.decode('utf-8')

with open('../output/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

X_full = scaler.transform(df_raw.drop(columns=['ID_REF', 'Tissue']))
print("Full shape:", X_full.shape)
assert (df_raw['Tissue'].values == y_true).all(), "Redosled instanci se ne poklapa!"
print("Redosled instanci potvrdjen - isti kao u PCA-50/TopVar-200 skupovima.")

In [ ]:
datasets = {
    'Full': X_full,
    'PCA-50': X_pca50,
    'TopVar-200': X_topvar,
}
for name, X in datasets.items():
    print(f"{name}: {X.shape}")

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

K_range = range(2, 11)
inertias = []
sils = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_pca50)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(X_pca50, labels))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(list(K_range), inertias, 'o-', color='#378ADD')
axes[0].set_xlabel('Broj klastera (K)')
axes[0].set_ylabel('Inercija')
axes[0].set_title('Elbow metoda')

axes[1].plot(list(K_range), sils, 'o-', color='#D85A30')
axes[1].set_xlabel('Broj klastera (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette analiza')

plt.tight_layout()
plt.savefig('../visualizations/elbow_silhouette.png', dpi=150)
plt.show()

best_k = list(K_range)[np.argmax(sils)]
print(f"Optimalan K po Silhouette metrici: {best_k}")
print(f"Napomena: Silhouette raste skoro monotono do kraja testiranog opsega, sto je "
      f"cesta pojava kod HDLSS podataka. Testiramo K={best_k} kao 'fini' izbor i K=2 "
      f"kao 'grubi' izbor (motivisan binarnom prirodom Tissue atributa, iako se ne "
      f"koristi direktno u klasterovanju).")

In [ ]:
from sklearn.cluster import AgglomerativeClustering, DBSCAN, Birch
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                              calinski_harabasz_score, adjusted_rand_score,
                              normalized_mutual_info_score)

def evaluate_clustering(name, dataset_name, X, labels, y_true):
    """Racuna interne i eksterne metrike za jedan rezultat klasterovanja.
    Eksterne metrike (ARI, NMI) porede se sa Tissue labelom ISKLJUCIVO radi
    naknadne interpretacije - labela se nigde ne koristi kao ulaz modelu."""
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = int(np.sum(labels == -1))
    mask = labels != -1

    if n_clusters < 2 or mask.sum() < 2:
        sil, db, ch = np.nan, np.nan, np.nan
    else:
        try:
            sil = silhouette_score(X[mask], labels[mask])
            db = davies_bouldin_score(X[mask], labels[mask])
            ch = calinski_harabasz_score(X[mask], labels[mask])
        except Exception:
            sil, db, ch = np.nan, np.nan, np.nan

    ari = adjusted_rand_score(y_true, labels)
    nmi = normalized_mutual_info_score(y_true, labels)

    return {
        'Algorithm': name, 'Dataset': dataset_name, 'K': n_clusters, 'Noise': n_noise,
        'Silhouette': sil, 'DaviesBouldin': db, 'CalinskiHarabasz': ch,
        'ARI_vs_Tissue': ari, 'NMI_vs_Tissue': nmi
    }
    

In [ ]:
all_labels = {}
results = []

for ds_name, X in datasets.items():
    t0 = time.time()
    print(f"--- Obrada skupa: {ds_name} (oblik {X.shape}) ---")

    # 1. KMeans K=2
    km2 = KMeans(n_clusters=2, random_state=RANDOM_STATE, n_init=10).fit(X)
    all_labels[(ds_name, 'KMeans (K=2)')] = km2.labels_
    results.append(evaluate_clustering('KMeans (K=2)', ds_name, X, km2.labels_, y_true))

    # 2. KMeans K=10
    km10 = KMeans(n_clusters=10, random_state=RANDOM_STATE, n_init=10).fit(X)
    all_labels[(ds_name, 'KMeans (K=10)')] = km10.labels_
    results.append(evaluate_clustering('KMeans (K=10)', ds_name, X, km10.labels_, y_true))

    # 3. Agglomerative Ward
    agg_w = AgglomerativeClustering(n_clusters=10, linkage='ward').fit(X)
    all_labels[(ds_name, 'Agglomerative (Ward)')] = agg_w.labels_
    results.append(evaluate_clustering('Agglomerative (Ward)', ds_name, X, agg_w.labels_, y_true))

    # 4. Agglomerative Complete
    agg_c = AgglomerativeClustering(n_clusters=10, linkage='complete').fit(X)
    all_labels[(ds_name, 'Agglomerative (Complete)')] = agg_c.labels_
    results.append(evaluate_clustering('Agglomerative (Complete)', ds_name, X, agg_c.labels_, y_true))

    # 5. DBSCAN - eps procenjen preko k-distance grafika (90-ti percentil 5-nn rastojanja)
    nn = NearestNeighbors(n_neighbors=5).fit(X)
    distances, _ = nn.kneighbors(X)
    k_dist = np.sort(distances[:, -1])
    eps_guess = np.percentile(k_dist, 90)
    dbs = DBSCAN(eps=eps_guess, min_samples=5).fit(X)
    all_labels[(ds_name, 'DBSCAN')] = dbs.labels_
    results.append(evaluate_clustering(f'DBSCAN (eps={eps_guess:.2f})', ds_name, X, dbs.labels_, y_true))

    # 6. BIRCH
    birch = Birch(n_clusters=10, threshold=0.5).fit(X)
    all_labels[(ds_name, 'BIRCH')] = birch.labels_
    results.append(evaluate_clustering('BIRCH', ds_name, X, birch.labels_, y_true))

    print(f"  Zavrseno za {time.time()-t0:.1f}s")

results_df = pd.DataFrame(results)
print("\nSvi eksperimenti zavrseni:", len(results_df), "kombinacija")


In [ ]:
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 20)
results_df_sorted = results_df.sort_values('Silhouette', ascending=False)
results_df_sorted.round(3)

In [ ]:
results_df.to_csv('../output/clustering_results.csv', index=False)

with open('../output/all_cluster_labels.pkl', 'wb') as f:
    pickle.dump(all_labels, f)

print("Sacuvano: output/clustering_results.csv, output/all_cluster_labels.pkl")
print("\nDAN 2 zavrsen. Spremno za Dan 3 - evaluacija i vizuelizacija.")
